# 07｜Swin Transformer 完整结构与复盘

现在把 Patch Embedding、Swin Blocks、Patch Merging 和分类头连接起来，得到完整的 Swin-T。

这一课结束后，原始 Swin Transformer 的核心理论主线就学习完成。

## 1. 第一步：图片变成 token 网格

输入是一张 224 × 224 的 RGB 图片。

Swin-T 使用 4 × 4 的 Patch Embedding，把图片切成 56 × 56 个不重叠区域，并把每个区域转换成 96 维 token。

输入从 B × 3 × 224 × 224 变成 B × 56 × 56 × 96。

与普通 ViT 不同，Swin 会继续保留 56 × 56 的二维网格结构，而不是只把它看成一条平坦序列。

## 2. Stage 1

Stage 1 包含两个 Swin Blocks：

1. 第一个 Block 使用 W-MSA，完成固定窗口内部交流。
2. 第二个 Block 使用 SW-MSA，完成跨原窗口边界交流。

整个 Stage 1 始终保持 B × 56 × 56 × 96。

Stage 内部更新特征内容，Stage 结束后的 Patch Merging 才改变 shape。

## 3. Stage 2

第一次 Patch Merging 把 B × 56 × 56 × 96 变成 B × 28 × 28 × 192。

Stage 2 同样包含两个 Blocks：先 W-MSA，后 SW-MSA。

空间位置减少了，但每个 token 的通道更多、对应的原图区域更大。

## 4. Stage 3

第二次 Patch Merging 得到 B × 14 × 14 × 384。

Stage 3 包含 6 个 Blocks，也就是三组 W-MSA 与 SW-MSA。

Swin-T 把最多的 Blocks 放在 Stage 3，因为此时空间尺寸已经降低，计算量更可控，同时 14 × 14 仍保留足够的空间信息。

## 5. Stage 4

第三次 Patch Merging 得到 B × 7 × 7 × 768。

Stage 4 包含两个 Blocks。此时每个 token 已经融合了很大的图片区域，特征更偏向整体语义。

Stage 4 后不再进行 Patch Merging。

## 6. Swin-T 四个 Stage 配置表

| Stage | 输出 shape | Block 数 | 注意力头数 | 窗口大小 |
|---|---|---:|---:|---|
| Stage 1 | B × 56 × 56 × 96 | 2 | 3 | 7 × 7 |
| Stage 2 | B × 28 × 28 × 192 | 2 | 6 | 7 × 7 |
| Stage 3 | B × 14 × 14 × 384 | 6 | 12 | 7 × 7 |
| Stage 4 | B × 7 × 7 × 768 | 2 | 24 | 7 × 7 |

通道数和 head 数同时翻倍，因此每个 head 分到的维度始终为 32。

## 7. 分类头怎样工作

Stage 4 输出 7 × 7，一共有 49 个空间 tokens。

Swin 不使用 ViT 的 CLS token，而是对 49 个 tokens 做全局平均池化：每个通道把 49 个位置取平均。

于是 B × 7 × 7 × 768 变成 B × 768。

最后使用线性分类层，把 768 维整图表示转换成各类别 logits。

## 8. 完整数据流

| 步骤 | 输出 shape |
|---|---|
| 输入图片 | B × 3 × 224 × 224 |
| Patch Embedding | B × 56 × 56 × 96 |
| Stage 1 | B × 56 × 56 × 96 |
| Patch Merging 1 | B × 28 × 28 × 192 |
| Stage 2 | B × 28 × 28 × 192 |
| Patch Merging 2 | B × 14 × 14 × 384 |
| Stage 3 | B × 14 × 14 × 384 |
| Patch Merging 3 | B × 7 × 7 × 768 |
| Stage 4 | B × 7 × 7 × 768 |
| 全局平均池化 | B × 768 |
| 分类头 | B × 类别数 |

## 9. Swin 为什么适合视觉任务

1. Window Attention 把计算限制在局部窗口，适合高分辨率特征。
2. Shifted Window 让窗口之间逐层通信。
3. 相对位置偏置告诉 Attention 两个 tokens 的方向和距离。
4. Patch Merging 形成从细节到语义的层级特征。
5. 多尺度输出适合继续用于分类、目标检测和语义分割等视觉任务。

## 10. CNN、ViT 与 Swin 的区别

| 对比 | CNN | ViT | Swin |
|---|---|---|---|
| 局部建模 | 卷积核 | 通常直接全局 Attention | Window Attention |
| 跨区域传播 | 堆叠卷积逐渐扩大 | 单层即可全局交互 | Shifted Window 逐层传播 |
| 层级下采样 | 池化或步长卷积 | 基础 ViT 通常保持 token 数 | Patch Merging |
| 空间关系 | 卷积结构天然包含 | 依赖位置编码 | 相对位置偏置 |
| 主要表示 | 特征图 | 平坦 token 序列 | 层级 token 网格 |

Swin 可以理解为：保留 Transformer 动态 Attention 的同时，引入了类似 CNN 的局部性和层级结构。

## 11. 从头到尾，每个模块解决什么问题

| 模块 | 解决的问题 |
|---|---|
| Patch Embedding | 图片怎样变成 token 网格 |
| W-MSA | 固定窗口内部怎样交流 |
| SW-MSA | 不同窗口之间怎样交流 |
| Attention Mask | 怎样删除循环移位产生的错误连接 |
| 相对位置偏置 | Attention 怎样知道方向和距离 |
| MLP | 每个 token 的内部特征怎样加工 |
| 残差连接 | 怎样保留原信息并稳定深层训练 |
| Patch Merging | 怎样下采样并建立层级特征 |
| 全局平均池化与分类头 | 怎样得到整张图片的类别输出 |

## 12. Swin Transformer 学习完成标准

如果能够独立回答下面这些问题，就已经完成原始 Swin Transformer 的核心理论学习：

1. 为什么全局 Attention 处理高分辨率图片较贵？
2. W-MSA 和 SW-MSA 分别怎样分组？
3. mask 在 Swin 中禁止什么连接？
4. 相对位置偏置为什么只有 169 种位移？
5. Patch Merging 怎样改变 H、W、C？
6. 一个完整 Swin Block 包含哪些部分？
7. Swin-T 四个 Stage 的 shape 怎样变化？
8. Swin 为什么不需要 CLS token？
9. Swin 与 CNN、ViT 的主要差异是什么？

## 13. 总结

Swin Transformer 的主线可以概括为：

图片先变成 token 网格；W-MSA 完成窗口内交流；SW-MSA 通过重新分组完成跨窗口交流；Patch Merging 逐层降低分辨率并增加通道；四个 Stage 形成层级特征；最后通过全局平均池化完成分类。

原始 Swin Transformer 的核心结构到这里已经讲完。后续若进入实践，应依次进行预训练推理、关键 shape 跟踪、分类头替换和小数据集微调，而不是继续增加理论名词。